# Baseline 1b: Tìm kiếm Từ khóa bằng Mô hình Okapi BM25

## 1. Lý thuyết toán học nâng cao (Dành cho Vấn đáp)

Okapi BM25 là thuật toán nâng cấp dựa trên nền tảng của TF-IDF, được coi là chuẩn mực công nghiệp cho các hệ thống tìm kiếm văn bản dạng từ khóa (Lexical Search).

### Công thức tính điểm Okapi BM25:
$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

#### Các thành phần chính trong công thức:
1. **$f(q_i, D)$**: Tần suất của từ truy vấn $q_i$ trong tài liệu $D$ (chính là Term Frequency).
2. **IDF của BM25**:
   $$\text{IDF}(q_i) = \log \frac{|D| - n(q_i) + 0.5}{n(q_i) + 0.5}$$
   *(Trong đó $n(q_i)$ là số tài liệu chứa từ $q_i$. Nếu từ khóa xuất hiện ở hơn một nửa số văn bản, IDF của BM25 có thể bị âm - đây là một điểm đặc trưng).* 

3. **Độ bão hòa tần suất - Term Frequency Saturation ($k_1$)**:
   - Tham số $k_1$ (thường chọn $1.2$ đến $2.0$) kiểm soát giới hạn bão hòa.
   - Trong TF-IDF, điểm số tăng tuyến tính với tần suất từ. Trong BM25, khi từ xuất hiện quá nhiều lần, điểm số sẽ tiệm cận một ngưỡng bão hòa để tránh việc một từ lặp lại bất thường thao túng kết quả.

4. **Chuẩn hóa độ dài tài liệu - Document Length Normalization ($b$)**:
   - Tham số $b$ (thường chọn $0.75$) kiểm soát mức độ phạt tài liệu dài.
   - $|D| / \text{avgdl}$ là tỷ lệ độ dài tài liệu so với độ dài trung bình toàn corpus.
   - Nếu tài liệu rất dài ($|D| \gg \text{avgdl}$), thuật toán sẽ phạt giảm điểm vì từ khóa có thể chỉ là ngẫu nhiên. Nếu tài liệu ngắn gọn, tập trung ($|D| \ll \text{avgdl}$), từ khóa khớp được đánh giá cao hơn.

## 2. Khởi tạo Toy Corpus (Dữ liệu mẫu)
Chúng dùng lại corpus mẫu giống file TF-IDF để dễ dàng đối chiếu kết quả.

In [5]:
import numpy as np
import pandas as pd
import re

corpus = {
    "doc1": "Apple Net sales in fiscal year 2023 were $383,285 million.",
    "doc2": "Apple payments for acquisition of property plant and equipment were $10,959 million.",
    "doc3": "Amazon net sales were $574,785 million in fiscal year 2023.",
    "doc4": "Amazon purchases of property and equipment in fiscal year 2023 were $52,729 million.",
    "doc5": "NVIDIA net income was $29,760 million in fiscal year 2024."
}

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s$]', '', text)
    return text.split()

tokenized_docs = {k: tokenize(v) for k, v in corpus.items()}
vocabulary = sorted(list(set(word for doc in tokenized_docs.values() for word in doc)))

print(f"[Log] Số lượng tài liệu: {len(tokenized_docs)}")
print(f"[Log] Tổng số từ vựng từ điển: {len(vocabulary)}")
print(f"Vocabulary: {vocabulary}")

[Log] Số lượng tài liệu: 5
[Log] Tổng số từ vựng từ điển: 28
Vocabulary: ['$10959', '$29760', '$383285', '$52729', '$574785', '2023', '2024', 'acquisition', 'amazon', 'and', 'apple', 'equipment', 'fiscal', 'for', 'in', 'income', 'million', 'net', 'nvidia', 'of', 'payments', 'plant', 'property', 'purchases', 'sales', 'was', 'were', 'year']


## 3. Lập chỉ mục BM25 thủ công (Step-by-Step BM25 Index Building)
Chúng ta tự tính toán các thông số trung bình của Corpus để nạp vào công thức.

In [6]:
# 3.1 Tính toán độ dài các tài liệu và độ dài trung bình (avgdl)
doc_lengths = {k: len(v) for k, v in tokenized_docs.items()}
avgdl = sum(doc_lengths.values()) / len(doc_lengths)

print("=== THÔNG SỐ ĐỘ DÀI TÀI LIỆU ===")
for k, l in doc_lengths.items():
    print(f"  - {k}: {l} từ (Tỷ lệ length/avgdl = {l/avgdl:.2f})")
print(f"\nĐộ dài trung bình corpus (avgdl): {avgdl:.2f} từ")

=== THÔNG SỐ ĐỘ DÀI TÀI LIỆU ===
  - doc1: 10 từ (Tỷ lệ length/avgdl = 0.91)
  - doc2: 12 từ (Tỷ lệ length/avgdl = 1.09)
  - doc3: 10 từ (Tỷ lệ length/avgdl = 0.91)
  - doc4: 13 từ (Tỷ lệ length/avgdl = 1.18)
  - doc5: 10 từ (Tỷ lệ length/avgdl = 0.91)

Độ dài trung bình corpus (avgdl): 11.00 từ


### 3.2 Tính toán ma trận IDF của BM25
Lưu ý công thức IDF của BM25 khác công thức của TF-IDF chuẩn.

In [7]:
bm25_idf = {}
num_docs = len(corpus)

for word in vocabulary:
    # n(q_i): số tài liệu chứa từ
    n_qi = sum(1 for tokens in tokenized_docs.values() if word in tokens)
    # Tính IDF theo công thức BM25
    idf = np.log((num_docs - n_qi + 0.5) / (n_qi + 0.5) + 1.0) # Tránh idf âm
    bm25_idf[word] = idf

df_bm25_idf = pd.Series(bm25_idf).sort_values(ascending=False)
print("[SUCCESS] Đã tính xong IDF của BM25!")
print("\n=== TOÀN BỘ IDF BM25 CỦA TỪ VỰNG ===")
df_bm25_idf_df = pd.DataFrame({'Từ': df_bm25_idf.index, 'IDF': df_bm25_idf.values})
print(df_bm25_idf_df.to_string(index=False))
df_bm25_idf_df

[SUCCESS] Đã tính xong IDF của BM25!

=== TOÀN BỘ IDF BM25 CỦA TỪ VỰNG ===
         Từ      IDF
     $10959 1.386294
     $29760 1.386294
    $383285 1.386294
     $52729 1.386294
    $574785 1.386294
       2024 1.386294
acquisition 1.386294
        was 1.386294
   payments 1.386294
     nvidia 1.386294
     income 1.386294
        for 1.386294
      plant 1.386294
  purchases 1.386294
   property 0.875469
      apple 0.875469
     amazon 0.875469
  equipment 0.875469
        and 0.875469
         of 0.875469
      sales 0.875469
       2023 0.538997
        net 0.538997
     fiscal 0.287682
       were 0.287682
         in 0.287682
       year 0.287682
    million 0.087011


,Từ,IDF
0,$10959,1.386294
1,$29760,1.386294
2,$383285,1.386294
3,$52729,1.386294
4,$574785,1.386294
5,2024,1.386294
6,acquisition,1.386294
7,was,1.386294
8,payments,1.386294
9,nvidia,1.386294


### 3.3 Dựng Toàn bộ Ma trận Trọng số BM25 (Document-Term Weight Matrix)
Tương tự TF-IDF, ta có thể tính trước trọng số BM25 cho mọi cặp (tài liệu, từ vựng) trong từ điển để có cái nhìn tổng quan về ma trận trọng số.

In [8]:
bm25_matrix = []
k1 = 1.2
b = 0.75
for doc_name, tokens in tokenized_docs.items():
    doc_len = len(tokens)
    doc_weights = {}
    for word in vocabulary:
        tf = tokens.count(word)
        if tf > 0:
            idf = bm25_idf[word]
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * (doc_len / avgdl))
            weight = idf * (numerator / denominator)
        else:
            weight = 0.0
        doc_weights[word] = weight
    bm25_matrix.append(doc_weights)

df_bm25 = pd.DataFrame(bm25_matrix, index=corpus.keys())
print("[SUCCESS] Đã xây dựng ma trận trọng số BM25!")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("\n=== TOÀN BỘ MA TRẬN TRỌNG SỐ BM25 ===")
print(df_bm25.to_string())
df_bm25

[SUCCESS] Đã xây dựng ma trận trọng số BM25!

=== TOÀN BỘ MA TRẬN TRỌNG SỐ BM25 ===
        $10959    $29760   $383285   $52729   $574785      2023      2024  acquisition    amazon       and     apple  equipment    fiscal       for        in    income   million       net    nvidia        of  payments     plant  property  purchases     sales       was      were      year
doc1  0.000000  0.000000  1.439842  0.00000  0.000000  0.559816  0.000000     0.000000  0.000000  0.000000  0.909285   0.000000  0.298794  0.000000  0.298794  0.000000  0.090372  0.559816  0.000000  0.000000  0.000000  0.000000  0.000000    0.00000  0.909285  0.000000  0.298794  0.298794
doc2  1.336587  0.000000  0.000000  0.00000  0.000000  0.000000  0.000000     1.336587  0.000000  0.844077  0.844077   0.844077  0.000000  1.336587  0.000000  0.000000  0.083891  0.000000  0.000000  0.844077  1.336587  1.336587  0.844077    0.00000  0.000000  0.000000  0.277367  0.000000
doc3  0.000000  0.000000  0.000000  0.00000  1.43

,$10959,$29760,$383285,$52729,$574785,2023,2024,acquisition,amazon,and,apple,equipment,fiscal,for,in,income,million,net,nvidia,of,payments,plant,property,purchases,sales,was,were,year
doc1,0.000000,0.000000,1.439842,0.00000,0.000000,0.559816,0.000000,0.000000,0.000000,0.000000,0.909285,0.000000,0.298794,0.000000,0.298794,0.000000,0.090372,0.559816,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.909285,0.000000,0.298794,0.298794
doc2,1.336587,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.336587,0.000000,0.844077,0.844077,0.844077,0.000000,1.336587,0.000000,0.000000,0.083891,0.000000,0.000000,0.844077,1.336587,1.336587,0.844077,0.00000,0.000000,0.000000,0.277367,0.000000
doc3,0.000000,0.000000,0.000000,0.00000,1.439842,0.559816,0.000000,0.000000,0.909285,0.000000,0.000000,0.000000,0.298794,0.000000,0.298794,0.000000,0.090372,0.559816,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.909285,0.000000,0.298794,0.298794
doc4,0.000000,0.000000,0.000000,1.29032,0.000000,0.501681,0.000000,0.000000,0.814859,0.814859,0.000000,0.814859,0.267766,0.000000,0.267766,0.000000,0.080988,0.000000,0.000000,0.814859,0.000000,0.000000,0.814859,1.29032,0.000000,0.000000,0.267766,0.267766
doc5,0.000000,1.439842,0.000000,0.00000,0.000000,0.000000,1.439842,0.000000,0.000000,0.000000,0.000000,0.000000,0.298794,0.000000,0.298794,1.439842,0.090372,0.559816,1.439842,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,1.439842,0.000000,0.298794


## 4. Thực thi Tìm kiếm và Chấm điểm BM25
Cài đặt hàm tính điểm BM25 cho mỗi tài liệu với tham số chuẩn $k_1 = 1.2$ và $b = 0.75$.

In [9]:
def search_bm25(query, k1=1.2, b=0.75):
    print(f"\n=== CÂU HỎI USER: '{query}' ===")
    query_tokens = tokenize(query)
    print(f"[Step 1] Tokenized Query: {query_tokens}")
    
    scores = {}
    for doc_name, tokens in tokenized_docs.items():
        score = 0.0
        doc_len = len(tokens)
        print(f"\n* Tính điểm BM25 cho {doc_name} (Độ dài L_d = {doc_len} từ, avgdl = {avgdl:.2f}):")
        
        for word in query_tokens:
            if word in vocabulary:
                tf = tokens.count(word)
                idf = bm25_idf[word]
                
                # Tính chi tiết từng phần của công thức BM25
                ratio = doc_len / avgdl
                len_penalty = 1 - b + b * ratio
                numerator = tf * (k1 + 1)
                denominator = tf + k1 * len_penalty
                term_score = idf * (numerator / denominator)
                
                score += term_score
                print(f"  - Từ '{word}': tf = {tf}, idf = {idf:.4f}")
                print(f"    + Tỷ lệ độ dài tài liệu L_d / avgdl = {doc_len} / {avgdl:.2f} = {ratio:.4f}")
                print(f"    + Phần phạt độ dài (1 - b + b * (L_d/avgdl)) = 1 - {b} + {b} * {ratio:.4f} = {len_penalty:.4f}")
                print(f"    + Tử số (tf * (k1 + 1)) = {tf} * {k1+1:.1f} = {numerator:.4f}")
                print(f"    + Mẫu số (tf + k1 * phạt) = {tf} + {k1} * {len_penalty:.4f} = {denominator:.4f}")
                print(f"    + Điểm thành phần = idf * (tử / mẫu) = {idf:.4f} * ({numerator:.4f} / {denominator:.4f}) = {term_score:.4f}")
            else:
                print(f"  - Từ '{word}': Không có trong từ điển (OOV)")
                
        scores[doc_name] = score
        print(f"  => TỔNG ĐIỂM BM25 CHO {doc_name} = {score:.4f}")
        
    # Sắp xếp xếp hạng
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\n=== KẾT QUẢ XẾP HẠNG BM25 RETRIEVAL ===")
    for rank, (doc_name, score) in enumerate(sorted_scores):
        print(f"  Hạng {rank+1}: {doc_name} | Score BM25: {score:.4f} | Nội dung: {corpus[doc_name]}")
        
    return sorted_scores

## 5. Kiểm thử và Đối chiếu Điểm yếu (Failure Analysis)

Bây giờ, chúng ta chạy hai câu hỏi để đối chiếu.

In [10]:
# Test 1: Khớp từ khóa
search_bm25("Apple Net sales in 2023")


=== CÂU HỎI USER: 'Apple Net sales in 2023' ===
[Step 1] Tokenized Query: ['apple', 'net', 'sales', 'in', '2023']

* Tính điểm BM25 cho doc1 (Độ dài L_d = 10 từ, avgdl = 11.00):
  - Từ 'apple': tf = 1, idf = 0.8755
    + Tỷ lệ độ dài tài liệu L_d / avgdl = 10 / 11.00 = 0.9091
    + Phần phạt độ dài (1 - b + b * (L_d/avgdl)) = 1 - 0.75 + 0.75 * 0.9091 = 0.9318
    + Tử số (tf * (k1 + 1)) = 1 * 2.2 = 2.2000
    + Mẫu số (tf + k1 * phạt) = 1 + 1.2 * 0.9318 = 2.1182
    + Điểm thành phần = idf * (tử / mẫu) = 0.8755 * (2.2000 / 2.1182) = 0.9093
  - Từ 'net': tf = 1, idf = 0.5390
    + Tỷ lệ độ dài tài liệu L_d / avgdl = 10 / 11.00 = 0.9091
    + Phần phạt độ dài (1 - b + b * (L_d/avgdl)) = 1 - 0.75 + 0.75 * 0.9091 = 0.9318
    + Tử số (tf * (k1 + 1)) = 1 * 2.2 = 2.2000
    + Mẫu số (tf + k1 * phạt) = 1 + 1.2 * 0.9318 = 2.1182
    + Điểm thành phần = idf * (tử / mẫu) = 0.5390 * (2.2000 / 2.1182) = 0.5598
  - Từ 'sales': tf = 1, idf = 0.8755
    + Tỷ lệ độ dài tài liệu L_d / avgdl = 10 / 11.

[('doc1', np.float64(3.236996724322915)),
 ('doc3', np.float64(2.3277115979725123)),
 ('doc5', np.float64(0.858610363564984)),
 ('doc2', np.float64(0.8440774280463896)),
 ('doc4', np.float64(0.7694469796563125))]

In [11]:
# Test 2: Gặp từ đồng nghĩa tài chính (Lexical Gap)
search_bm25("Amazon capital expenditures 2023")


=== CÂU HỎI USER: 'Amazon capital expenditures 2023' ===
[Step 1] Tokenized Query: ['amazon', 'capital', 'expenditures', '2023']

* Tính điểm BM25 cho doc1 (Độ dài L_d = 10 từ, avgdl = 11.00):
  - Từ 'amazon': tf = 0, idf = 0.8755
    + Tỷ lệ độ dài tài liệu L_d / avgdl = 10 / 11.00 = 0.9091
    + Phần phạt độ dài (1 - b + b * (L_d/avgdl)) = 1 - 0.75 + 0.75 * 0.9091 = 0.9318
    + Tử số (tf * (k1 + 1)) = 0 * 2.2 = 0.0000
    + Mẫu số (tf + k1 * phạt) = 0 + 1.2 * 0.9318 = 1.1182
    + Điểm thành phần = idf * (tử / mẫu) = 0.8755 * (0.0000 / 1.1182) = 0.0000
  - Từ 'capital': Không có trong từ điển (OOV)
  - Từ 'expenditures': Không có trong từ điển (OOV)
  - Từ '2023': tf = 1, idf = 0.5390
    + Tỷ lệ độ dài tài liệu L_d / avgdl = 10 / 11.00 = 0.9091
    + Phần phạt độ dài (1 - b + b * (L_d/avgdl)) = 1 - 0.75 + 0.75 * 0.9091 = 0.9318
    + Tử số (tf * (k1 + 1)) = 1 * 2.2 = 2.2000
    + Mẫu số (tf + k1 * phạt) = 1 + 1.2 * 0.9318 = 2.1182
    + Điểm thành phần = idf * (tử / mẫu) = 0.5390 

[('doc3', np.float64(1.4691012344075283)),
 ('doc4', np.float64(1.3165407216036695)),
 ('doc1', np.float64(0.5598161080571258)),
 ('doc2', np.float64(0.0)),
 ('doc5', np.float64(0.0))]

### Nhận xét lỗi trên Test 2:
- Tương tự như TF-IDF, **BM25 vẫn thất bại hoàn toàn (điểm số 0.0000)** khi tìm kiếm cụm từ đồng nghĩa 'capital expenditures' trong tài liệu Amazon.
- **Kết luận:** Dù Okapi BM25 tối ưu hóa tần suất từ và chiều dài tài liệu rất tốt, bản chất của nó vẫn là một bộ khớp từ khóa (Lexical Matcher). Do đó, nó bất lực trước khoảng cách từ nghĩa (Lexical Gap).